# Demo 12h — A nonlinear series and a stack of frozen recurrent encoders

**Class 12 · F1–F2** (slides: `slides/12_neural_networks.md`)

* **F1:** the Lorenz series (one coordinate, noisy): a nonlinear deterministic system. Ridge regression, a dense network and the three recurrent layers forecast 10 steps ahead (3 seeds). The climate series of Demo 12e was almost linear: ridge won there.
* **F2:** a meta model. The RNN, the LSTM and the GRU are trained, then frozen; their last hidden states are concatenated and a small MLP head is trained on them (all in Keras 3).

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import keras
import numpy as np
import polars as pl
from sklearn.linear_model import Ridge

from faa import demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEEDS = (0, 1, 2)
LOOK, HOR = 24, 10

## Slide: A series a line cannot forecast (F1)

The Lorenz system, $\dot x = 10(y - x)$, $\dot y = x(28 - z) - y$, $\dot z = xy - \tfrac{8}{3} z$, integrated with Runge-Kutta; we observe only $x$ every 0.1 time units, add noise of 0.05 (the series has standard deviation 1), and forecast $x$ ten steps (one time unit) ahead from the last 24 values.
20 000 points: 70 / 15 / 15 % chronological split. Metric: root mean square error (RMSE) on the last 15 %.

In [ ]:
Xtr, ytr, Xva, yva, Xte, yte = neural.lorenz_windows(20000, LOOK, HOR)


def rmse(p):
    return float(np.sqrt(np.mean((np.ravel(p) - yte) ** 2)))


ridge = Ridge(1.0).fit(Xtr[:, :, 0], ytr)
slide("The Lorenz series")
print(f"{len(Xtr)} train, {len(Xva)} validation, {len(Xte)} test windows of {LOOK} values; target {HOR} steps ahead")
print(
    f"standard deviation of the target {yte.std():.2f}; persistence RMSE {rmse(Xte[:, -1, 0]):.3f}; ridge on the 24 values RMSE {rmse(ridge.predict(Xte[:, :, 0])):.3f}"
)
fig_series, ax = plotlib.fig("full", 1.5)
t = np.arange(300)
ax.plot(t, Xtr[:300, 0, 0], color=BLUE, lw=0.8, label="$x_t$ (noisy)")
ax.axvspan(100, 100 + LOOK, color=SKY, alpha=0.3, label="one window")
ax.plot([100 + LOOK + HOR - 1], [ytr[100]], "o", color=RED, ms=4, label="its target")
ax.set_xlabel("step $t$")
ax.set_ylabel("$x$ (standardised)")
ax.legend(frameon=False, ncol=3, fontsize=6, loc="lower right")
plotlib.save(fig_series, "f1_series")

## Slide: Dense network against the recurrent layers (F1)

Each model has about the same size (32 units); early stopping on the validation part; the table averages 3 seeds.

In [ ]:
results = {"persistence": [rmse(Xte[:, -1, 0])], "ridge on the window": [rmse(ridge.predict(Xte[:, :, 0]))]}
models = {"rnn": {}, "lstm": {}, "gru": {}}
stop = lambda: [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]  # noqa: E731


def train(model):
    model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=30, batch_size=64, verbose=0, callbacks=stop())
    return model


results["MLP on the window"] = []
for s in SEEDS:
    results["MLP on the window"].append(
        rmse(train(neural.mlp_regressor(Xtr.shape[1:], seed=s)).predict(Xte, verbose=0))
    )
print("MLP on the window:", np.round(results["MLP on the window"], 3))

In [ ]:
results["RNN"] = []
for s in SEEDS:
    models["rnn"][s] = train(neural.recurrent_regressor("rnn", Xtr.shape[1:], 32, seed=s))
    results["RNN"].append(rmse(models["rnn"][s].predict(Xte, verbose=0)))
print("RNN:", np.round(results["RNN"], 3))

In [ ]:
results["LSTM"] = []
for s in SEEDS:
    models["lstm"][s] = train(neural.recurrent_regressor("lstm", Xtr.shape[1:], 32, seed=s))
    results["LSTM"].append(rmse(models["lstm"][s].predict(Xte, verbose=0)))
print("LSTM:", np.round(results["LSTM"], 3))

In [ ]:
results["GRU"] = []
for s in SEEDS:
    models["gru"][s] = train(neural.recurrent_regressor("gru", Xtr.shape[1:], 32, seed=s))
    results["GRU"].append(rmse(models["gru"][s].predict(Xte, verbose=0)))
print("GRU:", np.round(results["GRU"], 3))

In [ ]:
rows = [{"model": k, "RMSE": float(np.mean(v)), "std over seeds": float(np.std(v))} for k, v in results.items()]
slide("Forecasting the Lorenz series")
show(pl.DataFrame(rows))
print(
    f"the best recurrent layer has {rows[-1]['RMSE'] / rows[1]['RMSE']:.2f} times the RMSE of ridge (last row against second row)"
)
fig_rmse, ax = plotlib.fig("col", 1.7)
ax.barh(
    [r["model"] for r in rows][::-1],
    [r["RMSE"] for r in rows][::-1],
    xerr=[r["std over seeds"] for r in rows][::-1],
    color=[BLUE] * 4 + [GREY] * 2,
)
ax.set_xlabel("test RMSE")
plotlib.save(fig_rmse, "f1_forecast")
fig_trace, ax = plotlib.fig("col", 1.7)
sl = slice(200, 320)
ax.plot(yte[sl], color="black", lw=1.0, label="truth")
ax.plot(ridge.predict(Xte[:, :, 0])[sl], color=GREY, lw=0.8, label="ridge")
ax.plot(models["gru"][0].predict(Xte[sl], verbose=0).ravel(), color=BLUE, lw=0.8, label="GRU")
ax.set_xlabel("test window")
ax.set_ylabel("$x_{t+10}$")
ax.legend(frameon=False, fontsize=6, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.22))
plotlib.save(fig_trace, "f1_trace")

## Slide: Freeze the encoders, train an MLP head (F2)

For each seed the three trained regressors lose their last `Dense` layer: what is left is an **encoder** that maps a window to 32 numbers (the last hidden state).
The three encoders are **frozen** (`trainable = False`), their outputs are concatenated (96 numbers) and a head is trained on top: `Dense(32, relu)` and `Dense(1)`.
We compare it with the average of the three forecasts and with a linear head (no hidden layer).

In [ ]:
stack_rmse, linear_rmse, avg_rmse = [], [], []
for s in SEEDS:
    trio = [models[k][s] for k in ("rnn", "lstm", "gru")]
    before = [w.copy() for w in (np.array(v) for m in trio for v in m.get_weights())]
    avg_rmse.append(rmse(np.mean([m.predict(Xte, verbose=0).ravel() for m in trio], 0)))
    head = neural.frozen_stack(trio, hidden=(32,), seed=s)
    train(head)
    stack_rmse.append(rmse(head.predict(Xte, verbose=0)))
    lin = neural.frozen_stack(trio, hidden=(), seed=s)
    train(lin)
    linear_rmse.append(rmse(lin.predict(Xte, verbose=0)))
    after = [np.array(v) for m in trio for v in m.get_weights()]
    assert all(np.array_equal(a, b) for a, b in zip(before, after))
slide("The stacked model")
print(
    f"total weights {head.count_params()}, trainable {sum(int(np.prod(w.shape)) for w in head.trainable_weights)} (the head: 96 x 32 + 32 + 32 + 1)"
)
print("the weights of the three encoders are unchanged after training the head (checked)")
stack_rows = [{"model": r["model"], "RMSE": r["RMSE"], "std over seeds": r["std over seeds"]} for r in rows[3:]]
stack_rows += [
    {"model": "average of the three", "RMSE": float(np.mean(avg_rmse)), "std over seeds": float(np.std(avg_rmse))},
    {
        "model": "frozen + linear head",
        "RMSE": float(np.mean(linear_rmse)),
        "std over seeds": float(np.std(linear_rmse)),
    },
    {"model": "frozen + MLP head", "RMSE": float(np.mean(stack_rmse)), "std over seeds": float(np.std(stack_rmse))},
]
show(pl.DataFrame(stack_rows))
fig_stack, ax = plotlib.fig("col", 1.9)
ax.barh(
    [r["model"] for r in stack_rows][::-1],
    [r["RMSE"] for r in stack_rows][::-1],
    xerr=[r["std over seeds"] for r in stack_rows][::-1],
    color=[BLUE, BLUE, BLUE, GREY, ORANGE, GREEN][::-1],
)
ax.set_xlim(0.3, 0.45)
ax.set_xlabel("test RMSE")
plotlib.save(fig_stack, "f2_stack")